# Natural Language Processing (NLP): from text treatement to Transformers

For text treatment we have the following NNs options:

- *RNNs*: sequential treatment of text, but tend to vanishing or exploding gradient, because weights are multiplied many many times.
- *LSTM*: complex alternative to RNNs that can discard or retain certain informations, avoiding for vanishing or exploding gradient.
- *Transformers*: the fastest and achieving the best results.

**Transformers-based models** are the most emploied nowadays, based on
- parallel training;
- sequential generating;
- attention mechanism.

## 1) Representing the text for NNs

NNs receive tensors and not really strings of carachters. Models for language processing learn the text as follows:

> -> *Text* 
>
> --> *Tokens* (a single carachter, a piece of word, a word) 
>
> ---> *Integer numbers identification* (every token is associated to a number, for example Unicode) 
>
> ----> *Vector*.

> The NLPs receive the text and "embeddigs" it into a vector.

A table
'nn.Embedding(V, C)
'
represents the tokens C embedded into a vector V.

### Example:

Consider an English text on 26 letters in minuscule font, and
- BOS: token at the beginning of word.
- EOS: token at the end of word.


In [ ]:
import urllib.request

chars = list('abcdefghijklmnopqrstuvwyz')
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch}

In [3]:
example = torch.tensor(encode_name('emma'))
table = nn.Embedding(vocab_size, 16)
vectors = table(example)

print('Identifiants :', example.shape)
print('Vecteurs :', vectors.shape)
print('les deux m ont le meme embedding: ', torch.equal(vectors[2], vectors[3]))


NameError: name 'torch' is not defined

## 2) Prediction of the next token

**Goal of NLP**: learning the regularities among the tokens, to be able to predict on an unseen text the next token.

>A *language model* learns the conditional distribution on the vocabularity:
>$$
>P(x_{t+1} \mid x_{0},\dots, x_{1}),
>$$
>i.e. the probability of having token $x_{t+1}$ after tokens $x_{0},\dots,x_{t}$.

Then, once this conditional probability is known, it is possible to predict the next tokens on an unseen sequence of tokens / text.

> **Parallel Training:** we have a text / a sequence of tokens and we perform "teacher forcing", i.e. we predict SIMULTANEOUSLY / IN PARALLEL the next token for each given token.

> **Sequential Generation**: the predicted text on an unseen sequence is outputed sequentially one token at a time.

Rmk) The predictive power and strength of Transformers relies on the parallel training.

Rmk) *Transformers are trained on mini-batches, where each train is performed in parallel on the tokens.*

### Example:

**For example**, to generate a name, we start from BOS and choose a token, then we compute the conditionaly probability of having such a token given BOS.

## 3) The attention mechanism

How informations and relations among the various text tokens are collected? These connections between the tokens of a text are established by the attention mechanism.

>During the training phase, networks rely on **attention**:
>
>*each tokens influence each other; each token influence its neightbors*.

**Self attention** if the attention mechanism is addressed to the same text.

>The **attention** is a non-linear layer that produces a score for each token by its position, its neightbors, and its frequence in the text.

>EACH TOKEN produces 3 vecteurs (that do not depend by the position of the token):
>
>**Q, query**: what information from the position of the token.
>
>**K, key**: what key informations contains the token.
>
>**V, value**: what importance to give to the token.

The score $q_{t} k_{s}$ mesures the compatibility between two positions: position $q_{t}$ for content $k_{s}$.

The value $v_{t}$

>The **attention matrix** is:
>$$
>A = softmax\left( \frac{Q K^{T}}{\sqrt{d}}\right)
>$$
>and the **weighted attention matrix** is then:
>$$
>Z = AV
>$$.

The softmax entries refer to the other tokens.

Mask....

>**Multihead attention**: attention mechanism directed towards different "areas of attention"; like for example the grammar, the style, the language of a text.

Rmk) RNNs (recurrent neural networks) pass the informations from the left to the right. But, this mechanism is prone to gradient vanishing and therefore to the lost of information. Transformers are a great substitution of RNNs, learning instead through the attention mechanism.

## 4) MicroGPT Transformer

 > -> Tokens + positions
 >
 > --> Block transformers
 >
 > ---> Logits on the vocabulary

### TRAINING PHASE (encoder):

>***Architecture***
>
>INPUT 
>
>0) Tokenized text (in entry)
>
>------
>PRE-PROCESSING
>1) Token embedding
>
>2) Positional embedding
>
>------
>TRANSFORMER BLOCK (repeated many many times)
>
>3) MultiHead attention
>
>4) Layer Normalization
>
>5) Feed Forward NN == MLP
>
>-----
>OUTPUT 
>
>6) Output layer

2) Positional embedding: encoding tokens positions.

The attention mechanism does not care about the position of token, therefore it is necessary at the very beginning to encode the position of tokens.


4) Layer Normalisation: normalise the tokens, to make weight all the tokens the same.
$$
RMS(x)=\dfrac{}{}
$$

In [ ]:
def rms_norm(x):
    return x * torch.sqrt(x.square().mean(dim=-1, keepdim=True))

5. MLP: typically a 1 hidden layer feedforward neural network with a ReLu, and linear ouput.

It does not mix the positions of the tokens.

In [ ]:
class MicroGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)

In [ ]:
model_microgpt = MicroGPT().to(device)

optimizer = torch.optim.Adam(model.parameters())

In [ ]:
def micro_validation():
    was_training = model.training
    model.eval()
    losses = []
    for name in valid_examples:
        x,y = name_tensors(name)
        losses.append(F.cross_entropy(model(x), y).item())
    model.train(was_training)
    return float(np.mean(losses))

print('Validation before training:', round(micro_validation(), 3))

In [ ]:
losses, validation_curve = [], []
sampling_rng = random.Random(42)
for step in range(steps):

### GENERATION PHASE (decoder):

> - The network reads one token at a time,
>
>- and predicts the probability of having the other tokens after this first one.
>
>- then, it ouputs the token with higher probability.

For example, it reads 'BOS' and predicts the probability that after 'BOS' there is 'BOS', 'A', 'B', 'C', 'D', .... 'Z', and 'EOS'. It ouputs the token with higher probability (?).

>- The ouput token is subjected again, and generation begins again.

For example, let's say that after 'BOS' we predict 'B' with higher probability. Then, ''EOS, B'' are given in input to the network, and probability of find the other tokens after these two are computed again.

## 5) Subtokens

> Actual transformers focus on *subtokens*, instaed than *tokens / words*.

Instead than using single words as tokens, transformers focus on pieces of words. Focusing on subtokens takes into account:

- rare words;
- person names;
- flexions;
- neologisms.

This permits to eventually considering carachters as separate items, than to merge adjacent tokens depending on the probability, and so on, until real words are generated.

*Example*: starts from 'l' 'o' 'w', then merges 'lo' 'w', and finally 'low'.

## 6) Padding

> In this context, consists in adding some zeros after the last subtoken taken in input.

## 7) Document classifying

> Document classifying: predicting the subject of a text.